# 한식 LoRA 학습 노트북

SDXL-Turbo 기반 한식 특화 LoRA를 학습합니다.
학습 완료 후 결과물(.safetensors)을 Google Drive에 저장하고,
generate_images.ipynb에서 자동으로 불러와 사용합니다.

## 사전 준비
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. AI Hub(aihub.or.kr)에서 다운로드한 한식 이미지를
   Google Drive의 `/MyDrive/stockpipeline/lora_dataset/` 폴더에 업로드
   - 학습할 음식 카테고리별로 폴더 구성 불필요
   - 깨끗한 단일 요리 사진 30~100장 권장
   - 권장 해상도: 512x512 이상

## 예상 시간
- 환경 설정: 5분
- 모델 로드: 3분  
- 학습 (1000 steps, 50장 기준): 약 60~90분
- 결과 저장: 2분


In [ ]:
# 1. 환경 설치
!pip install -q diffusers transformers accelerate peft bitsandbytes
!pip install -q datasets Pillow torch torchvision

import os, json, shutil
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
print("Google Drive 마운트 완료")


In [ ]:
# 2. 설정
# ── 여기를 필요에 따라 수정하세요 ──────────────────────
DATASET_DIR = "/content/drive/MyDrive/stockpipeline/lora_dataset"
OUTPUT_DIR  = "/content/drive/MyDrive/stockpipeline/lora_output"

# 학습할 음식의 트리거 워드 (프롬프트에서 이 단어로 한식 스타일을 불러옴)
TRIGGER_WORD = "koreanfood"

# 학습 설정
TRAIN_STEPS    = 1000   # 데이터 30장: 800~1000, 50장 이상: 1000~1500
LEARNING_RATE  = 1e-4
LORA_RANK      = 16     # 4~32, 높을수록 용량 커지고 특화도 높아짐
BATCH_SIZE     = 1      # T4 VRAM 한계상 1 권장
SAVE_STEPS     = 500    # 중간 저장 주기
# ────────────────────────────────────────────────────────

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# 데이터셋 확인
imgs = list(Path(DATASET_DIR).glob("**/*.jpg")) + \
       list(Path(DATASET_DIR).glob("**/*.png")) + \
       list(Path(DATASET_DIR).glob("**/*.jpeg"))
print(f"발견된 이미지: {len(imgs)}장")
if len(imgs) < 20:
    print("⚠️  이미지가 20장 미만입니다. 품질이 낮을 수 있습니다.")
else:
    print("✅ 학습 준비 완료")


In [ ]:
# 3. 이미지 전처리 (512x512 크롭 + 캡션 생성)
from PIL import Image
import shutil

WORK_DIR = Path("/content/lora_train_data")
WORK_DIR.mkdir(exist_ok=True)

processed = 0
for img_path in imgs:
    try:
        img = Image.open(img_path).convert("RGB")
        # 중앙 크롭 후 512x512 리사이즈
        w, h = img.size
        min_side = min(w, h)
        left = (w - min_side) // 2
        top  = (h - min_side) // 2
        img = img.crop((left, top, left + min_side, top + min_side))
        img = img.resize((512, 512), Image.LANCZOS)

        out_path = WORK_DIR / f"img_{processed:04d}.jpg"
        img.save(out_path, "JPEG", quality=95)

        # 캡션 파일 생성 (트리거워드 + 음식 묘사)
        caption_path = WORK_DIR / f"img_{processed:04d}.txt"
        caption_path.write_text(
            f"{TRIGGER_WORD}, korean food, single dish, "
            f"food photography, white background, top-down view"
        )
        processed += 1
    except Exception as e:
        print(f"스킵: {img_path.name} ({e})")

print(f"전처리 완료: {processed}장")
print(f"저장 위치: {WORK_DIR}")


In [ ]:
# 4. SDXL-Turbo 모델 및 LoRA 설정
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model

MODEL_ID = "stabilityai/sdxl-turbo"

print("모델 로드 중... (2~3분 소요)")
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipe = pipe.to("cuda")

# LoRA 설정
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=["to_q", "to_k", "to_v", "to_out.0",
                    "proj_in", "proj_out",
                    "ff.net.0.proj", "ff.net.2"],
    lora_dropout=0.1,
    bias="none",
)

unet = get_peft_model(pipe.unet, lora_config)
unet.print_trainable_parameters()
print("\n✅ 모델 로드 및 LoRA 설정 완료")


In [ ]:
# 5. 학습
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim import AdamW
from diffusers import DDPMScheduler
import time

class FoodDataset(Dataset):
    def __init__(self, data_dir):
        self.imgs = sorted(Path(data_dir).glob("*.jpg"))
        self.txts = sorted(Path(data_dir).glob("*.txt"))
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]).convert("RGB")
        caption = self.txts[idx].read_text()
        return self.transform(img), caption

dataset = FoodDataset(WORK_DIR)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
optimizer = AdamW(unet.parameters(), lr=LEARNING_RATE)

unet.train()
pipe.text_encoder.requires_grad_(False)
pipe.text_encoder_2.requires_grad_(False)
pipe.vae.requires_grad_(False)

print(f"학습 시작: {TRAIN_STEPS} steps, 데이터 {len(dataset)}장")
start = time.time()
global_step = 0
losses = []

while global_step < TRAIN_STEPS:
    for imgs_batch, captions in loader:
        if global_step >= TRAIN_STEPS:
            break

        imgs_batch = imgs_batch.to("cuda", dtype=torch.float16)

        # 텍스트 인코딩
        tokens = pipe.tokenizer(
            captions, padding="max_length",
            max_length=pipe.tokenizer.model_max_length,
            truncation=True, return_tensors="pt"
        ).input_ids.to("cuda")
        tokens2 = pipe.tokenizer_2(
            captions, padding="max_length",
            max_length=pipe.tokenizer_2.model_max_length,
            truncation=True, return_tensors="pt"
        ).input_ids.to("cuda")

        with torch.no_grad():
            enc1 = pipe.text_encoder(tokens, output_hidden_states=True)
            enc2 = pipe.text_encoder_2(tokens2, output_hidden_states=True)
            text_emb = torch.concat([
                enc1.hidden_states[-2], enc2.hidden_states[-2]
            ], dim=-1)
            pooled = enc2[0]

            # VAE 인코딩
            latents = pipe.vae.encode(imgs_batch).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor

        # 노이즈 추가
        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0, noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],), device="cuda"
        ).long()
        noisy = noise_scheduler.add_noise(latents, noise, timesteps)

        # 예측 및 손실
        added_cond = {
            "text_embeds": pooled,
            "time_ids": torch.zeros(
                latents.shape[0], 6, device="cuda", dtype=torch.float16
            )
        }
        pred = unet(noisy, timesteps, text_emb,
                    added_cond_kwargs=added_cond).sample
        loss = torch.nn.functional.mse_loss(pred, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step()

        global_step += 1
        losses.append(loss.item())

        if global_step % 100 == 0:
            avg_loss = sum(losses[-100:]) / 100
            elapsed = time.time() - start
            eta = (elapsed / global_step) * (TRAIN_STEPS - global_step)
            print(f"Step {global_step}/{TRAIN_STEPS} | loss={avg_loss:.4f} | "
                  f"경과 {elapsed/60:.1f}분 | 남은 시간 {eta/60:.1f}분")

        if global_step % SAVE_STEPS == 0:
            ckpt = Path(OUTPUT_DIR) / f"checkpoint-{global_step}"
            unet.save_pretrained(str(ckpt))
            print(f"  → 중간 저장: {ckpt}")

print(f"\n✅ 학습 완료! 총 {(time.time()-start)/60:.1f}분 소요")


In [ ]:
# 6. LoRA 저장 (Google Drive + GitHub)
import requests

# LoRA 가중치 저장
lora_path = Path(OUTPUT_DIR) / "korean_food_lora"
unet.save_pretrained(str(lora_path))
print(f"✅ LoRA 저장: {lora_path}")

# GitHub에도 push (generate_images.ipynb에서 자동 로드)
GITHUB_REPO  = "stockpipeline/stockpipeline"
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

safetensors_file = lora_path / "adapter_model.safetensors"
if safetensors_file.exists():
    with open(safetensors_file, "rb") as f:
        content = f.read()
    import base64
    encoded = base64.b64encode(content).decode()

    # GitHub API로 업로드
    api_url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/lora/korean_food_lora.safetensors"
    res = requests.get(api_url, headers={"Authorization": f"Bearer {GITHUB_TOKEN}"})
    sha = res.json().get("sha") if res.status_code == 200 else None

    payload = {
        "message": "feat: add korean food LoRA weights",
        "content": encoded,
    }
    if sha:
        payload["sha"] = sha

    res = requests.put(api_url,
        headers={"Authorization": f"Bearer {GITHUB_TOKEN}"},
        json=payload
    )
    if res.status_code in [200, 201]:
        print("✅ GitHub push 완료: lora/korean_food_lora.safetensors")
    else:
        print(f"⚠️  GitHub push 실패 ({res.status_code}). Drive에만 저장됨.")
else:
    print("⚠️  safetensors 파일을 찾지 못했습니다.")

print(f"\nDrive 저장 위치: {lora_path}")
print("다음: generate_images.ipynb를 실행하면 자동으로 이 LoRA가 적용됩니다.")


In [ ]:
# 7. 학습 결과 테스트
# LoRA가 얼마나 잘 학습됐는지 바로 확인

from diffusers import AutoPipelineForText2Image

print("테스트 이미지 생성 중...")

test_prompts = [
    f"A single bowl of yukgaejang korean spicy beef soup, {TRIGGER_WORD}, top-down view, white background",
    f"A single bowl of bibimbap korean rice bowl, {TRIGGER_WORD}, top-down view, white background",
    f"A single bowl of kimchi-jjigae korean kimchi stew, {TRIGGER_WORD}, top-down view, white background",
]

for prompt in test_prompts:
    seed = 42
    generator = torch.Generator(device="cuda").manual_seed(seed)
    image = pipe(
        prompt=prompt,
        negative_prompt="multiple bowls, many dishes, grid, tiled, collage",
        num_inference_steps=4,
        guidance_scale=1.5,
        width=1024, height=1024,
        generator=generator,
    ).images[0]
    print(f"\n{prompt[:60]}...")
    display(image)
